# Chunk 가져와서 자르기

In [ ]:
import json
import re

import pandas as pd
from posthog.ai.openai import OpenAI

In [ ]:
with open('../Data_Files/chunk_split_list.json', 'r', encoding='utf-8') as f:
    chunk_wanted_list = json.load(f)
chunk_wanted_list

In [ ]:
chunk_wanted_list = chunk_wanted_list[39:]

In [ ]:
chunk_wanted_list[1:5]

In [ ]:
# chunk_wanted_list를 4분할
chunk_wanted_list_1 = chunk_wanted_list[:len(chunk_wanted_list)//4]
chunk_wanted_list_2 = chunk_wanted_list[len(chunk_wanted_list)//4:len(chunk_wanted_list)//2]
chunk_wanted_list_3 = chunk_wanted_list[len(chunk_wanted_list)//2:3*len(chunk_wanted_list)//4]
chunk_wanted_list_4 = chunk_wanted_list[3*len(chunk_wanted_list)//4:]


In [ ]:
print(len(chunk_wanted_list_1))
print(len(chunk_wanted_list_2))
print(len(chunk_wanted_list_3))
print(len(chunk_wanted_list_4))

# 청크 기준 기초적인 QA 페어 생성

In [ ]:
import os
import openai
import json
import re

# OpenAI API 키 설정 (환경 변수에서 읽어옵니다)
openai.api_key = os.getenv("OPENAI_API_KEY")
if not openai.api_key:
    raise ValueError("환경 변수 OPENAI_API_KEY에 API 키를 설정해주세요.")

# # Alpaca식 포맷을 위한 프롬프트 요소
# instruction_template = (
#     "당신은 채용 공고에 대한 질문과 답변을 생성하는 AI 비서입니다.\n"
#     "다음에 주어진 구인 공고 청크(chunk)를 보고, 총 5개의 질문-답변 페어를 생성하세요.\n"
#     "질문 유형은 아래와 같습니다:\n"
#     "1) 기본 질문 1개\n"
#     "2) 중간 난이도 질문 2개\n"
#     "3) 복잡한 질문 2개\n"
#     "– ‘마감일’, ‘채용절차’ 등 부수적 정보 제외\n"
#     "– 질문에 “이 회사”, “이 직무” 대신 **포지션 제목**을 반드시 포함하세요.  \n"
#     "  예: “JrBackend Engineer로 입사하면 어떤 기술 스택을 사용하나요?”\n"
#     "– 출력은 JSON 배열, 각 항목에 `question`과 `answer` 필드를 갖도록 작성해주세요."
# )

instruction = (
    "당신은 채용 공고에 대한 질문과 답변을 생성하는 AI 비서입니다.\n"
    "다음에 주어진 구인 공고 청크(chunk)를 보고, 총 5개의 질문-답변 페어를 생성하세요.\n"
    "질문은 반드시 아래의 5가지 토픽을 **한 번씩** 다루어야 합니다:\n"
    "  1) **직무 책임**: ‘[포지션]로서 주로 어떤 업무를 수행하나요?’\n"
    "  2) **기술 요건**: ‘[포지션]에서 사용하는 핵심 기술 스택은 무엇인가요?’\n"
    "  3) **경험 요건**: ‘이 포지션에 지원하려면 어떤 경력을 갖추어야 하나요?’\n"
    "  4) **회사 비전·문화**: ‘이 회사(공고에 나온 회사명)의 핵심 가치나 비전은 무엇인가요?’\n"
    "  5) **복지·혜택**: ‘이 포지션 지원자가 누릴 수 있는 주요 복지 혜택은 무엇인가요?’\n"
    "– 질문에 절대로 ‘이 회사’, ‘이 직무’ 같은 추상 대명사만 쓰지 말고, “JrBackend Engineer로서…”, “UIUX 디자이너로서…”처럼 **직무명**을 명시하세요.\n"
    "– ‘마감일’, ‘채용절차’ 등 부수적인 정보는 제외합니다.\n"
    "– 출력은 JSON 배열, 각 원소에 `question`과 `answer` 필드를 갖도록 작성해주세요."
)



results = []

for chunk in chunk_wanted_list_1:
    # 프롬프트 구성
    prompt = (
        f"### Instruction:\n{instruction_template}\n"
        f"### Input:\n{chunk}\n"
        "### Response:\n"
    )

    # 모델 호출
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    content = response.choices[0].message.content.strip()

    # 코드 블록 제거 및 JSON 추출
    # 트리플 백틱으로 감싸진 JSON을 추출
    match = re.search(r"```(?:json)?\n(.*)```", content, re.S)
    if match:
        json_str = match.group(1).strip()
    else:
        json_str = content

    # JSON 파싱
    try:
        qa_list = json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"JSON 디코드 오류: {e}\n원본 응답:\n{content}")
        continue

    # Alpaca 형식으로 재구성: instruction=chunk, input=질문, output=답변
    for qa in qa_list:
        question = qa.get('question')
        answer = qa.get('answer')
        if question and answer:
            results.append({
                "instruction": chunk,
                "input": question,
                "output": answer
            })

# JSON 파일로 저장
output_path = "alpaca_prompts_1.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"Alpaca 형식 데이터 ({len(results)}건) '{output_path}'에 저장 완료")


# QA 페어 개선

[1/10] 원본 QA 개선 중...
[1/10] 질문 변형 생성 중...


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

# QA 페어 다양화 시키기

In [ ]:
def toImproveText(text):
    if isinstance(text, list):  # 입력이 리스트인지 확인
        cleaned_list = []
        for item in text:
            cleaned_item = item.replace('\r', '').replace('\t', '')
            # +, #, :, %, ,는 유지하면서 나머지 특수문자 제거
            cleaned_item = re.sub(r'[^a-zA-Z0-9가-힣\s+#%:,]', '', cleaned_item).strip()
            cleaned_list.append(cleaned_item)
        return cleaned_list
    elif isinstance(text, int):
        return text
    else:
        cleaned_text = text.replace('\r', '').replace('\t', '')
        cleaned_text = re.sub(r'[^a-zA-Z0-9가-힣\s+#%:,]', '', cleaned_text).strip()
        return cleaned_text

In [ ]:
import random

random.shuffle(chunk_wanted_list)

chunk_wanted_list

In [ ]:
full_qa_list = []   # 전체 저장용
split_qa_list = []  # 2개마다 저장용
count = 1

for chunk_wanted_data_text in chunk_wanted_list:
    chunk_wanted_data_cleanedText = toImproveText(chunk_wanted_data_text)
    result = generate_qa_pairs(chunk_wanted_data_cleanedText)

    full_qa_list.append(result)     
    split_qa_list.append(result)    
    
    print(f"==================== {count}번째 완료 ====================")
    print(f"원문:\n{chunk_wanted_data_cleanedText}")
    print(f"{result}")

    if count == 1001: 
        break

    count += 1

In [ ]:
qa_list = []

for i in range(len(full_qa_list)):
    result = full_qa_list[i]
    chunk_text = chunk_wanted_list[i]

    qa_pairs = re.findall(r"Q\d+:\s*(.*?)\nA\d+:\s*(.*?)(?=\nQ\d+:|\Z)", result, re.DOTALL)

    for q, a in qa_pairs:
        qa_list.append({
            "instruction": q.strip(),
            "input": chunk_text.strip(),
            "output": a.strip()
        })

with open('chunk_qa_set_20250515_all.json', 'w', encoding='utf-8') as f:
    json.dump(qa_list, f, ensure_ascii=False, indent=2)

print("✅ 전체 JSON 저장 완료: chunk_qa_set_20250515_all.json")


In [ ]:
# csv 파일로 저장
csv_rows = []

for title, full_qa_list in qa_dict.items():
    for qa in full_qa_list:
        csv_rows.append({
            "질문": qa["질문"],
            "답변": qa["답변"]
        })

chunk_qa_set_df = pd.DataFrame(csv_rows)
chunk_qa_set_df.to_csv("chunk_qa_set_20250515_{count}.csv", index=False, encoding='utf-8-sig')
print("✅ 전체 csv 저장 완료: qa_set_20250515_all.csv")